# Klasifikasi DemogPairs Menggunakan ViT (Emosi dan Umur) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(emotion_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

  0%|          | 0/10800 [00:00<?, ?it/s]

  8%|▊         | 829/10800 [00:00<00:01, 8279.84it/s]

 15%|█▌        | 1657/10800 [00:00<00:01, 7267.27it/s]

 23%|██▎       | 2432/10800 [00:00<00:01, 7470.69it/s]

 30%|███       | 3251/10800 [00:00<00:00, 7740.63it/s]

...

 82%|████████▏ | 8893/10800 [00:01<00:00, 7770.02it/s]

 90%|████████▉ | 9677/10800 [00:01<00:00, 7789.35it/s]

 97%|█████████▋| 10458/10800 [00:01<00:00, 7791.20it/s]

100%|██████████| 10800/10800 [00:01<00:00, 7740.69it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',

}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_gnb_vit-emotion-age_',
    results_path='results/demogpairs_gnb_vit-emotion-age_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB


{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.001603718743751331), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}


Accuracy  : 0.7680555555555556
Precision : 0.7685712235997165
Recall    : 0.7680555555555556
F1 Score  : 0.7680515833672463
               precision    recall  f1-score   support

Asian_Females     0.7542    0.7417    0.7479       360
  Asian_Males     0.7253    0.7333    0.7293       360
Black_Females     0.7762    0.7806    0.7784       360
  Black_Males     0.8092    0.7778    0.7932       360
White_Females     0.7874    0.7611    0.7740       360
  White_Males     0.7591    0.8139    0.7855       360

     accuracy                         0.7681      2160
    macro avg     0.7686    0.7681    0.7681      2160
 weighted avg     0.7686    0.7681    0.7681      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9166666666666666,0.7542372881355932,0.7416666666666667,0.7478991596638657,360
Asian_Males,0.9092592592592592,0.7252747252747253,0.7333333333333333,0.7292817679558011,360
Black_Females,0.9259259259259259,0.7762430939226519,0.7805555555555556,0.778393351800554,360
Black_Males,0.9324074074074075,0.8092485549132948,0.7777777777777778,0.7932011331444758,360
White_Females,0.9259259259259259,0.7873563218390804,0.7611111111111111,0.7740112994350282,360
White_Males,0.9259259259259259,0.7590673575129534,0.8138888888888889,0.7855227882037534,360


Confusion matrix saved: images\cm_gnb_vit-emotion-age_GaussianNB.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               267                29                19                 7                34                 4
         Asian_Males                32               264                 6                21                 5                32
       Black_Females                26                 6               281                22                21                 4
         Black_Males                 1                22                26               280                 1                30
       White_Females                24                 7                29                 3               274                23
         White_Males                 4                36                 1                13                13               293


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
GaussianNB,models/clf_demogpairs_gnb_vit-emotion-age_GaussianNB.pkl,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.001603718743751331), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7680555555555556,0.7680515833672463,0.7685712235997165,0.7680555555555556,240


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-emotion-age_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 1788.0,
 'days': 0,
 'hours': 0,
 'minutes': 29,
 'seconds': 48.0,
 'text': '0 hari 0 jam 29 menit 48.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 10867.0,
 'days': 0,
 'hours': 3,
 'minutes': 1,
 'seconds': 7.0,
 'text': '0 hari 3 jam 1 menit 7.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.001603718743751331), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7465,0.7436,0.7535,0.7633,0.7703,0.7554,0.7553,0.7568,0.7554,10.0442
2,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00307029062975785), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7459,0.7436,0.7535,0.7598,0.7714,0.7549,0.7547,0.7563,0.7549,13.5744
3,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0008376776400682924), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7454,0.7431,0.7529,0.7622,0.7697,0.7546,0.7545,0.7559,0.7546,14.6115
4,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(1e-09), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7471,0.7419,0.7541,0.761,0.7679,0.7544,0.7542,0.7556,0.7544,22.4602
...,...,...,...,...,...,...,...,...,...,...,...
237,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': None, 'scaler': 'MinMaxScaler'}",0.5498,0.5422,0.526,0.5365,0.5498,0.5409,0.5342,0.5834,0.5409,2.2583
238,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(27.283333764867695), 'pca': 'PCA', 'scaler': None}",0.5382,0.5422,0.526,0.5405,0.5538,0.5402,0.5387,0.5848,0.5402,8.2921
239,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(52.233450742668325), 'pca': 'PCA', 'scaler': None}",0.537,0.5376,0.5249,0.5382,0.5538,0.5383,0.5366,0.5838,0.5383,7.2615
240,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': 'PCA', 'scaler': None}",0.5347,0.5382,0.5237,0.5353,0.5521,0.5368,0.5349,0.5823,0.5368,8.6639
